# 00 - Planteamiento de Negocio
**Proyecto:** K-asar — Stellar Data Clustering
**Dataset:** Sloan Digital Sky Survey - Data Release 17 (SDSS DR17)

**Fase:** Contexto de Negocio (previa al Análisis Exploratorio de Datos)

Este notebook precede a `01_exploratory_data_analysis.ipynb` y establece el marco de negocio bajo el cual se toman todas las decisiones técnicas del resto del pipeline: qué problema resolvemos, para quién, con qué datos, y qué se espera entregar al final.


---
## 1. El cliente y el encargo

**Consultora:** DataScope Solutions — división de analítica avanzada.

**Cliente:** Meridian Sky Survey Consortium, un consorcio de observación astronómica que opera un telescopio de barrido fotográfico continuo del cielo nocturno.

El consorcio nos contrata porque acumula, cada noche, cientos de miles de nuevas detecciones fotométricas — pero **no tiene forma de saber, en el momento de la detección, qué tipo de objeto acaba de fotografiar**. Confirmar el tipo de objeto requiere un paso adicional, mucho más costoso: la espectroscopía.


---
## 2. El problema de negocio

Un survey como el SDSS mide, para cada objeto que fotografía, cinco magnitudes de brillo en distintas bandas de color ($u, g, r, i, z$). Esta fotometría es **barata**: se obtiene automáticamente, para millones de objetos, en cada barrido del cielo.

Pero saber si un objeto es una **estrella**, una **galaxia** o un **cuásar** (núcleo galáctico activo alimentado por un agujero negro supermasivo) requiere **espectroscopía**: apuntar el telescopio específicamente a ese objeto y descomponer su luz en un espectro detallado. Esto es **caro** — el tiempo de telescopio es un recurso escaso, compartido y muy disputado entre distintos equipos de investigación del consorcio.

**La pregunta de negocio que debemos responder:**

> ¿Podemos usar únicamente la fotometría barata — sin ninguna confirmación espectroscópica previa — para agrupar automáticamente los objetos detectados en categorías con significado astrofísico, de modo que el equipo científico priorice su tiempo de telescopio hacia los objetos más interesantes o ambiguos, en lugar de asignarlo a ciegas?

Este es, por diseño, un problema de **aprendizaje no supervisado**: un objeto recién fotografiado no tiene todavía una clase confirmada. El archivo histórico del SDSS sí contiene la clase real (`class`) para objetos que ya fueron observados por espectroscopía en el pasado — pero para construir una herramienta que funcione con objetos *nuevos*, el pipeline de modelado debe actuar exactamente como si esa etiqueta no existiera. Por eso, en `02_data_preprocessing.ipynb`, la columna `class` se aísla del conjunto de entrenamiento y se reserva únicamente para validar el resultado al final, en `05_business_translation_and_ethics.ipynb`.


---
## 3. Selección del dataset

Elegimos el **Stellar Classification Dataset (SDSS17)**, publicado por fedesoriano en Kaggle, disponible en:
https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17

**Por qué este dataset encaja con el problema:**
- Contiene datos reales del SDSS, con las cinco bandas fotométricas ($u,g,r,i,z$) que cualquier survey obtiene de forma barata y automática.
- Incluye también la clase real (`class`: `STAR`, `GALAXY`, `QSO`) y el `redshift`, confirmados por espectroscopía — el recurso caro — lo que nos permite **validar honestamente** el clustering no supervisado al final del proyecto, algo que no sería posible con un dataset sin ninguna verdad de referencia.
- Con 100,000 registros, tiene volumen suficiente para que los patrones descubiertos sean estadísticamente robustos, y para simular de forma realista el volumen de detecciones que un survey real acumula por noche.


In [ ]:
# Descarga del dataset (mismo mecanismo usado en 01_exploratory_data_analysis.ipynb)
import kagglehub
import os

dataset_dir = kagglehub.dataset_download("fedesoriano/stellar-classification-dataset-sdss17")
file_path = os.path.join(dataset_dir, "star_classification.csv")

import pandas as pd
df_preview = pd.read_csv(file_path)
print(f"Registros: {df_preview.shape[0]:,} | Columnas: {df_preview.shape[1]}")
df_preview.head(3)


---
## 4. Objetivos del proyecto

Alineados con el encargo original de la consultora:

1. **Selección y justificación del dataset** — completado en esta sección.
2. **Preprocesamiento riguroso** — gestionar valores nulos y erróneos (no confundir errores de instrumento con outliers astrofísicos reales), y codificar/escalar variables adecuadamente.
3. **Descubrimiento de patrones** — agrupar los objetos en segmentos lógicos mediante algoritmos de clustering no supervisado (K-Means, DBSCAN, GMM, Jerárquico), sin usar nunca la clase real durante el entrenamiento.
4. **Reducción de dimensionalidad** — simplificar el espacio de variables mediante PCA para su correcta interpretación y visualización.
5. **Evaluación y robustez** — validar matemáticamente la calidad de las agrupaciones, y contrastarlas —solo al final— contra la clase real como verdad de referencia.
6. **Propuesta de producto** — traducir los clusters descubiertos en una herramienta práctica de **triage de espectroscopía**: qué objetos observar primero con el recurso caro.


---
## 5. Estructura del proyecto y equipo

El trabajo se organiza en **6 notebooks**, cada uno con un responsable dentro del equipo:

| # | Notebook | Fase | Responsable |
|---|---|---|---|
| 00 | `00_business_case.ipynb` | Planteamiento de negocio | Javi |
| 01 | `01_exploratory_data_analysis.ipynb` | Análisis Exploratorio de Datos | Javi |
| 02 | `02_data_preprocessing.ipynb` | Preprocesamiento de Datos | Luis |
| 03 | `03_dimensionality_reduction.ipynb` | PCA y Selección de K | Isabella |
| 04 | `04_modeling.ipynb` | Clustering y Modelo | Josema |
| 05 | `05_business_translation_and_ethics.ipynb` | Traducción de Negocio, Demo y Ética | Yohanna |

Cada notebook consume los artefactos exportados por el anterior (`data/stellar_scaled.csv`, `data/X_pca.csv`, `data/k_optimo.json`, etc.), permitiendo que cada integrante trabaje y ejecute su fase de forma independiente sin repetir pasos ya resueltos.


---
## 6. Producto final esperado

Al cierre del proyecto, el equipo entrega:

1. **Código del proyecto** — los 6 notebooks de este repositorio, con documentación interna en celdas de Markdown.
2. **Repositorio en GitHub** (`github.com/yohperez/k-sar`) — estructura de carpetas modular (`notebooks/`, `data/`, `src/`, `assets/`), commits descriptivos de todo el equipo, y README explicativo.
3. **Presentación técnico-comercial** — un pitch de 10 a 12 minutos dirigido al cliente (Meridian Sky Survey Consortium), con una **demo en vivo** de la herramienta de triage desplegada en producción.

El resto de este repositorio desarrolla, paso a paso, cada uno de estos compromisos.
